In [1]:
%load_ext autoreload
%autoreload 2

In [16]:
import pandas as pd
import numpy as np
import plotly.express as px

from src.get_tables import (
    tickers, path_dim_tickers,
    get_dimension_table_tickers, get_historical_data_tickers, 
    get_historical_data_assets, get_historical_br_indexes
)

save_folder_path = 'dados/raw_trusted'

In [32]:
# Salvar tabelas
get_dimension_table_tickers(tickers, path_dim_tickers, save_folder_path)
get_historical_data_tickers(tickers, save_folder_path, start = '2016-06-20', end = '2026-06-20')
get_historical_data_assets(save_folder_path, start = '2016-06-20', end = '2026-06-20')
get_historical_br_indexes(save_folder_path, start = None)

In [33]:
# Reler tabelas 
dim_tickers = pd.read_parquet(f"{save_folder_path}/dim_tickers.parquet", engine = 'pyarrow')
df_hist_tickers = pd.read_parquet(f"{save_folder_path}/historical_data_tickers.parquet", engine = 'pyarrow')
df_ativos = pd.read_parquet(f"{save_folder_path}/historical_data_assets.parquet", engine = 'pyarrow')
df_br_indexes = pd.read_parquet(f"{save_folder_path}/historical_data_br_indexes.parquet", engine = 'pyarrow')

In [95]:
# Gerar tabela analítica
def get_analitical_table(df_hist_tickers, df_ativos, df_br_indexes, ticker):

    # juntar tabelas
    tmp = df_hist_tickers[df_hist_tickers['Ticker'] == ticker].copy()
    tmp = tmp.merge(df_ativos, on = 'Date', how = 'left')
    tmp = tmp.merge(df_br_indexes, on = 'Date', how = 'left')

    tmp.columns = tmp.columns.str.lower()

    # criar coluna de dia da semana e remover finais de semana
    tmp['weekday'] = tmp['date'].dt.day_name()
    tmp = tmp[~tmp['weekday'].isin(['Saturday', 'Sunday'])]

    # filtrar período de dados
    tmp = tmp[tmp['date'] >= pd.to_datetime('2016-07-01')]
    tmp = tmp[tmp['date'] < pd.to_datetime('2026-06-01')]

    # completar nulls
    tmp['selic'] = tmp['selic'].ffill()
    tmp['ipca'] = tmp['ipca'].ffill()

    for asset in ['sp_500', 'dolar', 'ibovespa']:
        tmp[f"close_{asset}"] = tmp[f"close_{asset}"].ffill()
        for metrica in ['amplitude_pct', 'var_dia_pct']:
            tmp[f"{metrica}_{asset}"] = tmp[f"{metrica}_{asset}"].fillna(0)

    # métricas adicionais

    # médias móveis
    tmp["ma5"] = tmp["close"].rolling(20).mean()
    tmp["ma30"] = tmp["close"].rolling(50).mean()

    # bollinger bands (volatilidade): usam média móvel de 20 períodos e ± 2 desvios padrão
    rolling = tmp["close"].rolling(20)
    std = rolling.std()
    tmp["bb_middle"] = rolling.mean()
    tmp["bb_upper"] = tmp["bb_middle"] + 2 * std
    tmp["bb_lower"] = tmp["bb_middle"] - 2 * std

    # RSI - relative strength index (ação muito comprada ou muito vendida) - varia de 0 a 100, < 30 é sobrevendida e > 70 é sobrecomprada
    delta = tmp["close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    tmp["rsi"] = 100 - (100 / (1 + rs))

    # RSI Wilder (média exponencial)
    delta = tmp["close"].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()

    rs = avg_gain / avg_loss

    tmp["rsi_wilder"] = 100 - (100 / (1 + rs))

    # MACD - moving average convergence divergence (diferença entre duas médias móveis exponenciais) - identificar mudanças de tendência, aceleração do movimento e desaceleração
    ema12 = tmp["close"].ewm(span=12, adjust=False).mean()
    ema26 = tmp["close"].ewm(span=26, adjust=False).mean()

    tmp["macd"] = ema12 - ema26

    tmp["macd_signal"] = (
        tmp["macd"]
        .ewm(span=9, adjust=False)
        .mean()
    )

    tmp["macd_hist"] = (
        tmp["macd"] - tmp["macd_signal"]
    )

    return tmp

In [96]:
tb_analitica = get_analitical_table(df_hist_tickers, df_ativos, df_br_indexes, ticker = 'ABEV3')
print(f"{tb_analitica.shape[0]:_}")

2_469


In [97]:
tb_analitica.head()

,date,ticker,close,high,low,open,volume,amplitude_pct,var_dia_pct,close_dolar,close_ibovespa,close_sp_500,amplitude_pct_dolar,amplitude_pct_ibovespa,amplitude_pct_sp_500,var_dia_pct_dolar,var_dia_pct_ibovespa,var_dia_pct_sp_500,selic,ipca,weekday,ma5,ma30,bb_middle,bb_upper,bb_lower,rsi,rsi_wilder,macd,macd_signal,macd_hist
9,2016-07-01,ABEV3,12.81,12.81,12.66,12.72,22773000.00,0.01,0.01,3.21,52233.00,2102.95,0.02,0.02,0.01,-0.00,0.01,0.00,0.05,0.52,Friday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,0.00
10,2016-07-04,ABEV3,12.70,12.86,12.65,12.81,6702100.00,0.02,-0.01,3.23,52569.00,2102.95,0.01,0.01,0.00,-0.00,0.01,0.00,0.05,0.52,Monday,NaN,NaN,NaN,NaN,NaN,NaN,0.00,-0.01,-0.00,-0.01
11,2016-07-05,ABEV3,12.86,12.88,12.64,12.64,13405900.00,0.02,0.02,3.29,51842.00,2088.55,0.01,0.02,0.01,0.00,-0.01,-0.00,0.05,0.52,Tuesday,NaN,NaN,NaN,NaN,NaN,NaN,9.96,-0.00,-0.00,-0.00
12,2016-07-06,ABEV3,12.78,12.85,12.64,12.82,11263900.00,0.02,-0.00,3.30,51902.00,2099.73,0.02,0.02,0.01,0.00,0.00,0.01,0.05,0.52,Wednesday,NaN,NaN,NaN,NaN,NaN,NaN,9.47,-0.00,-0.00,-0.00
13,2016-07-07,ABEV3,12.73,12.86,12.67,12.78,8916400.00,0.02,-0.00,3.33,52015.00,2097.90,0.01,0.02,0.01,0.00,0.00,-0.00,0.05,0.52,Thursday,NaN,NaN,NaN,NaN,NaN,NaN,9.12,-0.01,-0.00,-0.01


In [91]:
# import plotly.graph_objects as go

# fig = go.Figure()

# fig.add_trace(go.Line(x = tb_analitica['date'], y = tb_analitica['var_dia_pct'], name = 'ABEV3'))
# fig.add_trace(go.Line(x = tb_analitica['date'], y = tb_analitica['var_dia_pct_dolar'], name = 'Dólar'))
# fig.add_trace(go.Line(x = tb_analitica['date'], y = tb_analitica['var_dia_pct_ibovespa'], name = 'Ibovespa'))
# fig.add_trace(go.Line(x = tb_analitica['date'], y = tb_analitica['var_dia_pct_sp_500'], name = 'S&P 500'))
# fig.show()

In [94]:
# tb_analitica['weekday'].value_counts()

In [93]:
# fig = px.line(df_ativos[df_ativos['Ticker'] == 'Dolar'], x = 'Date', y = 'Close')
# fig.update_layout(plot_bgcolor = 'white', title = 'US$ para R$')

# fig.update_xaxes(
#     showgrid=True,
#     gridcolor="lightgray",
#     gridwidth=1
# )

# fig.update_yaxes(
#     showgrid=True,
#     gridcolor="lightgray",
#     gridwidth=1,
#     tickprefix="R$ ",
#     tickformat=",.2f",
#     range = (0, 7)
# )

# fig.show()